# The Anatomy of a Transformer: A Hands-On Guide

This notebook walks through **every component** of a GPT-2 transformer, using IRTK
to inspect real weights and activations. By the end, you'll have a concrete
understanding of:

1. **Token Embeddings** -- how text becomes vectors
2. **Positional Embeddings** -- how the model knows word order
3. **The Residual Stream** -- the central communication bus
4. **Layer Normalization** -- keeping activations stable
5. **Attention** -- how the model moves information between positions
6. **MLPs** -- where the model stores and retrieves knowledge
7. **Unembedding** -- how the model produces its final prediction

We'll use GPT-2 Small (124M parameters, 12 layers, 12 heads) as our specimen.

### Prerequisites
- Basic Python and NumPy familiarity
- No deep learning expertise required -- we'll explain everything!

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

from irtk import HookedTransformer

matplotlib.rcParams.update({'figure.facecolor': 'white', 'font.size': 11})

# Load GPT-2 Small
model = HookedTransformer.from_pretrained("gpt2")
cfg = model.cfg
print(f"GPT-2 Small: {cfg.n_layers} layers, {cfg.n_heads} heads, d_model={cfg.d_model}, d_head={cfg.d_head}")

---
## Part 1: Tokenization

Before anything else, text must be converted into **tokens** -- integer IDs that
index into the model's vocabulary. GPT-2 uses **Byte Pair Encoding (BPE)**, which
splits text into subword units. Common words are single tokens; rare words get
split into pieces.

The vocabulary has 50,257 tokens.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

text = "The cat sat on the mat."
tokens = tokenizer.encode(text)
print(f"Text: '{text}'")
print(f"Token IDs: {tokens}")
print(f"Decoded tokens:")
for i, tok_id in enumerate(tokens):
    tok_str = tokenizer.decode([tok_id])
    print(f"  Position {i}: token_id={tok_id:5d} -> '{tok_str}'")

print(f"\nVocabulary size: {cfg.d_vocab}")
print(f"\nNote: GPT-2 prefixes words with a space character (Ġ in BPE notation).")
print(f"'The' is one token, but ' cat' (with leading space) is also one token.")

---
## Part 2: Token Embeddings (W_E)

Each token ID is mapped to a **768-dimensional vector** by looking up a row in the
**embedding matrix** W_E. This matrix has shape `[50257, 768]` -- one row per token
in the vocabulary.

Think of it as: each token gets its own unique 768-number "fingerprint" that the
model has learned during training.

```
token_id = 464 ("The")  -->  W_E[464]  -->  [0.12, -0.34, 0.87, ..., 0.03]  (768 numbers)
```

In [ ]:
W_E = np.array(model.embed.W_E)
print(f"Embedding matrix W_E shape: {W_E.shape}")
print(f"  {W_E.shape[0]} tokens x {W_E.shape[1]} dimensions\n")

# Look up embeddings for our tokens
tokens_jax = jnp.array(tokens)
embeddings = W_E[tokens]  # numpy indexing with a list is fine
print(f"Embeddings for '{text}':")
print(f"  Shape: {embeddings.shape}  (seq_len x d_model)")
print(f"  First token ('The') embedding (first 10 dims): {embeddings[0, :10].round(3)}")

# Visualize: how similar are these embeddings to each other?
# Cosine similarity between all pairs
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
cosine_sim = (embeddings @ embeddings.T) / (norms @ norms.T)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cosine_sim, cmap='RdBu', vmin=-1, vmax=1)
tok_labels = [tokenizer.decode([t]) for t in tokens]
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tok_labels, rotation=45, ha='right')
ax.set_yticklabels(tok_labels)
ax.set_title('Cosine Similarity of Token Embeddings')
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{cosine_sim[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("Notice: 'The' and ' the' are somewhat similar -- the model has learned that")
print("they represent the same word despite different capitalization.")

---
## Part 3: Positional Embeddings (W_pos)

Attention treats input as a **set** -- it doesn't inherently know word order.
"The cat sat" and "sat cat The" would look the same without position information.

GPT-2 solves this by **adding** a positional embedding to each token embedding.
Position 0 gets one vector, position 1 gets another, etc. The model learns these
during training.

```
residual_stream[pos] = W_E[token_id] + W_pos[pos]
```

W_pos has shape `[1024, 768]` -- GPT-2 can handle sequences up to 1024 tokens.

In [ ]:
W_pos = np.array(model.pos_embed.W_pos)
print(f"Positional embedding matrix W_pos shape: {W_pos.shape}")
print(f"  {W_pos.shape[0]} positions x {W_pos.shape[1]} dimensions\n")

# Visualize: how do positional embeddings relate to each other?
# Nearby positions should be more similar
positions_to_show = [0, 1, 2, 5, 10, 50, 100, 500, 1000, 1023]
pos_embeds = W_pos[positions_to_show]
norms = np.linalg.norm(pos_embeds, axis=1, keepdims=True)
cos_sim = (pos_embeds @ pos_embeds.T) / (norms @ norms.T)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Cosine similarity between selected positions
im = ax1.imshow(cos_sim, cmap='RdBu', vmin=-1, vmax=1)
ax1.set_xticks(range(len(positions_to_show)))
ax1.set_yticks(range(len(positions_to_show)))
ax1.set_xticklabels(positions_to_show, fontsize=8)
ax1.set_yticklabels(positions_to_show, fontsize=8)
ax1.set_xlabel('Position')
ax1.set_ylabel('Position')
ax1.set_title('Cosine Similarity Between Position Embeddings')
plt.colorbar(im, ax=ax1)

# Right: First few dimensions of W_pos across all positions
for dim in range(4):
    ax2.plot(W_pos[:100, dim], label=f'Dim {dim}', alpha=0.7)
ax2.set_xlabel('Position')
ax2.set_ylabel('Value')
ax2.set_title('First 4 Dimensions of W_pos (positions 0-99)')
ax2.legend()

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - Nearby positions have similar embeddings (smooth gradient in similarity)")
print("  - Some dimensions show periodic patterns (like sinusoidal encodings)")
print("  - The model learned these patterns from data, not from a formula")

---
## Part 4: The Residual Stream

This is the **most important concept** in mechanistic interpretability.

The residual stream is a `[seq_len, 768]` tensor that starts as the sum of token
and position embeddings. Every layer **reads from** and **writes to** this stream:

```
residual = embed(tokens) + pos_embed(positions)   # Initial
residual = residual + attention_output(residual)   # Layer 0 attention writes
residual = residual + mlp_output(residual)         # Layer 0 MLP writes
residual = residual + attention_output(residual)   # Layer 1 attention writes
residual = residual + mlp_output(residual)         # Layer 1 MLP writes
...                                                # (12 layers total)
logits = unembed(layer_norm(residual))             # Final prediction
```

Each component **adds** its output. Nothing is overwritten. This means the final
residual stream is a **sum of all contributions** from all components.

This is why we can decompose it: the model's prediction = embedding + attn_0 +
mlp_0 + attn_1 + mlp_1 + ... + attn_11 + mlp_11.

In [ ]:
# Run the model and cache everything
tokens_jax = jnp.array(tokens)
logits, cache = model.run_with_cache(tokens_jax)

# The residual stream at different points
embed = np.array(cache['hook_embed'])
pos_embed = np.array(cache['hook_pos_embed'])
initial_residual = embed + pos_embed

print(f"Residual stream shape at every point: {initial_residual.shape}")
print(f"  = [{len(tokens)} tokens, {cfg.d_model} dimensions]\n")

# Show how the residual stream evolves
norms = []
labels = ['embed+pos']
norms.append(np.linalg.norm(initial_residual, axis=-1).mean())

for l in range(cfg.n_layers):
    resid_post = np.array(cache[('resid_post', l)])
    norms.append(np.linalg.norm(resid_post, axis=-1).mean())
    labels.append(f'after L{l}')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(norms)), norms, color='steelblue')
ax.set_xticks(range(len(norms)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean L2 Norm')
ax.set_title('Residual Stream Norm Through the Model')
plt.tight_layout()
plt.show()

print("The residual stream grows in norm as each layer adds its contribution.")
print("This is the 'residual' in 'residual stream' -- it accumulates information.")

---
## Part 5: Layer Normalization

Before each attention layer and each MLP, GPT-2 applies **Layer Normalization**.
This is a simple operation that:

1. Centers the vector (subtract the mean)
2. Scales to unit variance (divide by standard deviation)
3. Applies learned scale (w) and bias (b) parameters

```
LayerNorm(x) = w * (x - mean(x)) / std(x) + b
```

**Why does this matter for interpretability?** Layer norm makes it harder to
directly read off information from the residual stream, because it normalizes
away magnitude information. The `logit_lens` technique (projecting intermediate
residual streams through the unembedding) needs to account for this.

In [ ]:
# Inspect layer norm parameters
ln1_w = np.array(model.blocks[0].ln1.w)
ln1_b = np.array(model.blocks[0].ln1.b)
print(f"Layer 0, pre-attention LayerNorm:")
print(f"  w (scale) shape: {ln1_w.shape}, range: [{ln1_w.min():.3f}, {ln1_w.max():.3f}]")
print(f"  b (bias) shape: {ln1_b.shape}, range: [{ln1_b.min():.3f}, {ln1_b.max():.3f}]")

# Show the effect of layer norm on the residual stream
resid_pre = np.array(cache[('resid_pre', 0)])  # [seq_len, d_model]

# Manual layer norm to show the steps
mean = resid_pre.mean(axis=-1, keepdims=True)
var = resid_pre.var(axis=-1, keepdims=True)
normalized = (resid_pre - mean) / np.sqrt(var + 1e-5)
ln_output = normalized * ln1_w + ln1_b

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Before LN
axes[0].hist(resid_pre[0], bins=50, alpha=0.7, color='steelblue')
axes[0].set_title(f'Before LayerNorm (position 0)\nmean={resid_pre[0].mean():.2f}, std={resid_pre[0].std():.2f}')
axes[0].set_xlabel('Value')

# After centering + scaling
axes[1].hist(normalized[0], bins=50, alpha=0.7, color='orange')
axes[1].set_title(f'After normalize\nmean={normalized[0].mean():.4f}, std={normalized[0].std():.2f}')
axes[1].set_xlabel('Value')

# After w, b
axes[2].hist(ln_output[0], bins=50, alpha=0.7, color='green')
axes[2].set_title(f'After w*x+b\nmean={ln_output[0].mean():.2f}, std={ln_output[0].std():.2f}')
axes[2].set_xlabel('Value')

plt.suptitle('Layer Normalization Step by Step', fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 6: Attention -- The Heart of the Transformer

Attention is how the model **moves information between positions**. Each attention
head independently decides:
1. **What to look for** (queries, from W_Q)
2. **What to advertise** (keys, from W_K)
3. **What to send** (values, from W_V)
4. **How to write to the residual stream** (output, from W_O)

### The Attention Computation (for one head)

```
q = x @ W_Q[head]          # Each position creates a query: "I'm looking for..."
k = x @ W_K[head]          # Each position creates a key: "I contain..."
v = x @ W_V[head]          # Each position creates a value: "If you attend to me, here's what I'll send"

scores = q @ k.T / sqrt(d_head)  # How well does each query match each key?
pattern = softmax(scores)         # Normalize to get attention weights (sum to 1)
z = pattern @ v                   # Weighted sum of values
result = z @ W_O[head]            # Project back to residual stream
```

GPT-2 Small has **12 heads per layer**, each operating independently in 64
dimensions (12 x 64 = 768 = d_model). Their results are summed.

### Causal Masking
GPT-2 is **autoregressive** -- position i can only attend to positions 0..i.
Future positions are masked out before softmax.

In [ ]:
# Attention weights for our sentence
tok_labels = [tokenizer.decode([t]) for t in tokens]

# Show all 12 heads in layer 0
pattern_L0 = np.array(cache[('pattern', 0)])  # [n_heads, seq_len, seq_len]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for h in range(12):
    ax = axes[h // 4, h % 4]
    im = ax.imshow(pattern_L0[h], cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'L0 Head {h}', fontsize=9)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tok_labels, rotation=90, fontsize=6)
    ax.set_yticklabels(tok_labels, fontsize=6)

plt.suptitle('Attention Patterns: Layer 0, All 12 Heads\n'
             'Each cell shows how much the row position attends to the column position',
             fontsize=12)
plt.tight_layout()
plt.show()

print("Common patterns you'll see:")
print("  - 'Previous token' heads: strong diagonal just below the main diagonal")
print("  - 'BOS' heads: strong first column (everything attends to beginning)")
print("  - 'Local' heads: attention concentrated near the diagonal")
print("  - 'Sparse' heads: attending to specific tokens (like punctuation)")

In [ ]:
# Let's look at the actual Q, K, V vectors
q = np.array(cache[('q', 0)])  # [seq_len, n_heads, d_head]
k = np.array(cache[('k', 0)])
v = np.array(cache[('v', 0)])

print(f"Q shape: {q.shape}  = [seq_len, n_heads, d_head]")
print(f"K shape: {k.shape}")
print(f"V shape: {v.shape}")
print(f"\nEach position has {cfg.n_heads} query/key/value vectors, each {cfg.d_head}-dimensional.")
print(f"Total attention dimensions: {cfg.n_heads} x {cfg.d_head} = {cfg.n_heads * cfg.d_head} = d_model")

# Weight matrices
W_Q = np.array(model.blocks[0].attn.W_Q)  # [n_heads, d_model, d_head]
W_K = np.array(model.blocks[0].attn.W_K)
W_V = np.array(model.blocks[0].attn.W_V)
W_O = np.array(model.blocks[0].attn.W_O)  # [n_heads, d_head, d_model]

print(f"\nWeight shapes:")
print(f"  W_Q: {W_Q.shape}  (projects d_model -> d_head, per head)")
print(f"  W_K: {W_K.shape}")
print(f"  W_V: {W_V.shape}")
print(f"  W_O: {W_O.shape}  (projects d_head -> d_model, per head)")
print(f"\nTotal attention parameters per layer: {4 * cfg.n_heads * cfg.d_model * cfg.d_head:,}")

---
## Part 7: MLPs -- The Knowledge Store

After attention moves information between positions, the **MLP** (Multi-Layer
Perceptron) processes each position independently. It's a simple two-layer
network:

```
mlp_out = activation(x @ W_in + b_in) @ W_out + b_out
```

- `W_in`: [768, 3072] -- projects up to 4x the model dimension
- `activation`: GELU (a smooth version of ReLU)
- `W_out`: [3072, 768] -- projects back down

**Key insight for interpretability**: Individual MLP neurons can sometimes represent
interpretable concepts. The `hook_pre` activation (before GELU) and `hook_post`
activation (after GELU) let us inspect what each neuron does.

In [ ]:
# MLP weights
W_in = np.array(model.blocks[0].mlp.W_in)
W_out = np.array(model.blocks[0].mlp.W_out)
print(f"MLP weight shapes:")
print(f"  W_in:  {W_in.shape}  (d_model -> d_mlp, expands 4x)")
print(f"  W_out: {W_out.shape}  (d_mlp -> d_model, compresses 4x)")
print(f"  MLP hidden dim (d_mlp): {cfg.d_mlp}")
print(f"  MLP parameters per layer: {cfg.d_model * cfg.d_mlp * 2:,}")

# Look at MLP activations
mlp_pre = np.array(cache[('pre', 0)])   # Before activation: [seq_len, d_mlp]
mlp_post = np.array(cache[('post', 0)])  # After activation: [seq_len, d_mlp]

print(f"\nMLP activations shape: {mlp_pre.shape}  (seq_len x d_mlp)")
print(f"  {cfg.d_mlp} 'neurons', each producing a scalar for each token position")

# Visualize neuron activations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Show top activated neurons for each token
im1 = ax1.imshow(mlp_post[:, :50], cmap='viridis', aspect='auto')
ax1.set_xlabel('Neuron index (first 50)')
ax1.set_ylabel('Token position')
ax1.set_yticks(range(len(tokens)))
ax1.set_yticklabels(tok_labels, fontsize=8)
ax1.set_title('MLP Post-Activation (Layer 0, first 50 neurons)')
plt.colorbar(im1, ax=ax1)

# Distribution of activations
ax2.hist(mlp_post.flatten(), bins=100, alpha=0.7, color='steelblue')
ax2.set_xlabel('Activation value')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of MLP Activations (Layer 0)')
ax2.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\nFraction of neurons that are ~zero (< 0.01): {(np.abs(mlp_post) < 0.01).mean():.1%}")
print("GELU kills many neurons, making MLP representations sparse -- good for interp!")

---
## Part 8: Unembedding -- Making the Final Prediction

After all 12 layers, the final residual stream is projected through the
**unembedding matrix** W_U to produce a 50,257-dimensional vector of **logits**
(one per vocabulary token). Higher logits = higher probability.

```
logits = LayerNorm(residual_final) @ W_U + b_U
```

GPT-2 **ties** its embedding and unembedding weights: `W_U = W_E.T`. This means
the model's "understanding" of a token (embedding) and its "ability to predict"
that token (unembedding) share the same learned representation.

Applying softmax to logits gives probabilities.

In [ ]:
W_U = np.array(model.unembed.W_U)
print(f"Unembedding matrix W_U shape: {W_U.shape}  (d_model x d_vocab)")
print(f"Embedding matrix W_E shape:   {W_E.shape}  (d_vocab x d_model)")
print(f"W_U ≈ W_E.T: {np.allclose(W_U, W_E.T, atol=1e-5)}  (weight tying)\n")

# What does the model predict at each position?
logits_np = np.array(logits)
probs = np.exp(logits_np) / np.exp(logits_np).sum(axis=-1, keepdims=True)

print(f"Predictions (the model predicts the NEXT token at each position):\n")
for i in range(len(tokens)):
    top5 = np.argsort(logits_np[i])[-5:][::-1]
    actual_next = tokens[i+1] if i < len(tokens)-1 else None
    top5_str = [f"'{tokenizer.decode([t])}' ({probs[i, t]:.1%})" for t in top5]
    actual_str = f" [actual: '{tokenizer.decode([actual_next])}']" if actual_next else " [end]"
    print(f"  After '{tokenizer.decode([tokens[i]])}': {', '.join(top5_str)}{actual_str}")

---
## Part 9: Decomposing Contributions

Since the residual stream is a **sum** of all component outputs, we can decompose
the model's prediction into individual contributions.

For each component (embedding, attn_0, mlp_0, attn_1, mlp_1, ...), we can ask:
"How much did this component push the prediction toward the correct token?"

This is **logit attribution** -- the bread and butter of mechanistic interpretability.

In [ ]:
# Decompose the residual stream into contributions
decomp, labels = cache.decompose_resid(return_labels=True)
print(f"Decomposed into {len(labels)} components:")
for label in labels:
    print(f"  {label}")

# Compute logit attribution for the correct next token at each position
# Which components contribute most to predicting the right answer?
W_U_jax = model.unembed.W_U

# For the last position, what components contribute to the top prediction?
last_pos = len(tokens) - 1
top_pred = int(np.argmax(logits_np[last_pos]))
top_pred_str = tokenizer.decode([top_pred])

# Project each component's contribution through the unembedding for this token
attrs = []
for i in range(len(labels)):
    # Component contribution at last position, projected to the predicted token's logit
    component = np.array(decomp[i, last_pos])  # [d_model]
    W_U_np = np.array(W_U_jax)
    attr = component @ W_U_np[:, top_pred]
    attrs.append(float(attr))

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['green' if a > 0 else 'red' for a in attrs]
ax.bar(range(len(attrs)), attrs, color=colors, alpha=0.7)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([l.replace('blocks.', 'L').replace('.hook_', ' ') for l in labels],
                    rotation=45, ha='right', fontsize=7)
ax.set_ylabel(f"Logit contribution")
ax.set_title(f"Logit Attribution: Who contributes to predicting '{top_pred_str}'\n"
             f"at the last position (after '{tokenizer.decode([tokens[last_pos]])}')")
ax.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print(f"\nGreen = pushes toward '{top_pred_str}', Red = pushes away")
print(f"The prediction is the SUM of all these contributions.")

---
## Summary: The Full Picture

```
Input text: "The cat sat on the mat."
         |
    [Tokenizer: text -> token IDs]
         |
    [W_E: token IDs -> embeddings]  +  [W_pos: positions -> pos embeddings]
         |
    = Initial residual stream  (shape: [seq_len, 768])
         |
    For each of 12 layers:
      |-> LayerNorm -> Attention (12 heads) -> Add to residual
      |-> LayerNorm -> MLP (3072 neurons)    -> Add to residual
         |
    [Final LayerNorm]
         |
    [W_U: residual -> logits over 50,257 tokens]
         |
    [Softmax: logits -> probabilities]
         |
    Output: probability distribution over next token
```

### Key Takeaways for Interpretability

1. **The residual stream is a sum** -- every component writes additively, so we can
   decompose the prediction into individual contributions.

2. **Attention moves information** between positions -- by reading the attention
   patterns, we can see which tokens are "talking to" which.

3. **MLPs transform information** at each position -- individual neurons can
   represent interpretable features.

4. **Everything is differentiable** -- we can use gradients to attribute importance,
   and we can intervene (ablate, patch, modify) at any hook point.

5. **IRTK caches everything** -- `run_with_cache()` gives you Q, K, V, attention
   scores, patterns, MLP activations, and residual streams at every layer.

### Next Steps

- **`induction_heads.ipynb`**: Discover the induction circuit
- **`03_logit_lens.ipynb`**: See how predictions evolve layer by layer
- **`04_activation_patching.ipynb`**: Identify which components cause specific behaviors
- **`05_linear_probes.ipynb`**: Train classifiers on internal representations